In [1]:
# %% [markdown]
# # Comprehensive Model Evaluation on Test Data
# 
# This notebook loads all previously trained models and evaluates them on the master test set (`data/master_splits/test_set.parquet`). It performs the following steps for each model:
# 
# 1.  **Loads the required test data and labels.**
# 2.  **Loads the trained model artifacts** from the `models/` directory.
# 3.  **Generates predictions** on the unseen test data.
# 4.  **Calculates and displays key performance metrics** (Classification Report, Accuracy, ROC-AUC, etc.).
# 5.  **Generates and saves all evaluation plots** (Confusion Matrix, Feature Importance, etc.) to the `results/` directory.
# 
# This provides a complete, end-to-end performance summary of the entire trained pipeline.

In [2]:
# %%
# ===================================================================
# CELL 1: SETUP AND IMPORTS
# ===================================================================
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder

# --- Import all project-specific classes and configurations ---
# This assumes the notebook is in the root of your project directory.
from config import Config
from data_loader import (
    get_data_for_binary, get_data_for_multiclass, get_data_for_virus,
    load_master_test_set, IoTDataset
)
from binary_classifier import BinaryClassifier
from multiclass_classifier import MultiClassXGBoost, MultiClassNeuralNetModel
from virus_classifier import VirusClassifier
from autoencoder import AutoencoderModel
from autoencoder_denoising import DenoisingAutoencoderModel
from clustering import KMeansClustering

# Ensure the results directory exists
os.makedirs(Config.RESULTS_DIR, exist_ok=True)

# Set seeds and print mode info for consistency
Config.set_seeds()
Config.print_mode_info()

# ## 1. Binary Classification: Benign vs. Malicious
# 
# Here, we evaluate the performance of the tuned LightGBM model and the baseline logistic regression model on the binary task of identifying traffic as either benign or malicious.


ModuleNotFoundError: No module named 'lightgbm'

In [ ]:
# %%
# ===================================================================
# CELL 2: EVALUATE BINARY CLASSIFIERS
# ===================================================================
print("=" * 70)
print("Evaluating Binary Classification Models")
print("=" * 70)

# 1. Load Data
# The function conveniently splits the master set for us. We only need the test portion.
_, X_test_bin, _, y_test_bin = get_data_for_binary()

# 2. Evaluate Tuned LightGBM Model
print("\n--- Evaluating: Tuned LightGBM ---")
try:
    lgbm_classifier = BinaryClassifier()
    model_path = os.path.join(Config.MODELS_DIR, 'binary_classifier_tuned.pkl')
    model_data = joblib.load(model_path)
    lgbm_classifier.model = model_data['model']
    lgbm_classifier.best_params = model_data['params']
    
    print(f"✓ Model loaded from {model_path}")
    lgbm_results = lgbm_classifier.evaluate(X_test_bin, y_test_bin)
except FileNotFoundError:
    print(f"❌ ERROR: Model file not found at {model_path}. Please run binary_classifier.py first.")

# 3. Evaluate Baseline Logistic Regression Model
if Config.RUN_LOGISTIC_REGRESSION:
    print("\n--- Evaluating: Baseline LightGBM (Logistic Config) ---")
    try:
        lr_classifier = BinaryClassifier()
        model_path = os.path.join(Config.MODELS_DIR, 'binary_classifier_logistic.pkl')
        model_data = joblib.load(model_path)
        lr_classifier.model = model_data
        
        print(f"✓ Model loaded from {model_path}")
        # Pass save_plots=False to avoid overwriting the main model's plots if names are the same
        # For this project, plot names are distinct, so it's safe to set to True.
        lr_results = lr_classifier.evaluate(X_test_bin, y_test_bin, save_plots=True) 
    except FileNotFoundError:
        print(f"❌ ERROR: Model file not found at {model_path}. Please run binary_classifier.py first.")

# ## 2. Multi-Class Classification: Attack Type
# 
# Next, we evaluate the models trained to classify the specific type of attack (e.g., DDoS, PortScan). This includes both the XGBoost and the PyTorch Neural Network models.

In [ ]:
## ===================================================================
# CELL 3: EVALUATE MULTI-CLASS CLASSIFIERS
# ===================================================================
print("\n" + "=" * 70)
print("Evaluating Multi-Class Classification Models")
print("=" * 70)

# 1. Load Data
_, X_test_multi, _, y_test_multi, le_multi = get_data_for_multiclass()
print(f"\n✓ Loaded {len(X_test_multi):,} test samples for multi-class evaluation.")

# 2. Evaluate XGBoost Model
print("\n--- Evaluating: Multi-Class XGBoost ---")
try:
    xgb_model = MultiClassXGBoost()
    model_path = os.path.join(Config.MODELS_DIR, 'multiclass_xgboost.pkl')
    model_data = joblib.load(model_path)
    xgb_model.model = model_data['model']
    xgb_model.best_params = model_data['params']
    
    print(f"✓ Model loaded from {model_path}")
    xgb_results = xgb_model.evaluate(X_test_multi, y_test_multi, le_multi)
except FileNotFoundError:
    print(f"❌ ERROR: Model file not found at {model_path}. Please run multiclass_classifier.py first.")
    
# 3. Evaluate Neural Network Model
print("\n--- Evaluating: Multi-Class Neural Network ---")
try:
    input_dim = X_test_multi.shape[1]
    num_classes = len(le_multi.classes_)
    nn_model = MultiClassNeuralNetModel(input_dim, num_classes)
    
    model_path = os.path.join(Config.MODELS_DIR, 'multiclass_nn.pth')
    nn_model.load_model(model_path) # The class has its own robust loader
    
    print(f"✓ Model loaded from {model_path}")
    # The NN model expects numpy arrays, not DataFrames
    nn_results = nn_model.evaluate(X_test_multi.values, y_test_multi, le_multi)
except FileNotFoundError:
    print(f"❌ ERROR: Model file not found at {model_path}. Please run multiclass_classifier.py first.")

# ## 3. Virus Family Classification
# 
# This section focuses on the fine-grained task of identifying the specific malware family (e.g., Mirai, Hakai) for traffic already known to be malicious.

In [ ]:
# ===================================================================
# CELL 4: EVALUATE VIRUS FAMILY CLASSIFIER
# ===================================================================
print("\n" + "=" * 70)
print("Evaluating Virus Family Classification Model")
print("=" * 70)

# 1. Load Data
_, X_test_virus, _, y_test_virus, le_virus = get_data_for_virus()
print(f"\n✓ Loaded {len(X_test_virus):,} malicious test samples for virus family evaluation.")

# 2. Evaluate LightGBM Model
try:
    virus_classifier = VirusClassifier()
    model_path = os.path.join(Config.MODELS_DIR, 'virus_classifier.pkl')
    model_data = joblib.load(model_path)
    virus_classifier.model = model_data['model']
    virus_classifier.best_params = model_data['params']
    
    print(f"✓ Model loaded from {model_path}")
    virus_results = virus_classifier.evaluate(X_test_virus, y_test_virus, le_virus)
    
    print("\n--- Analyzing prediction confidence per family ---")
    virus_classifier.analyze_families(X_test_virus, y_test_virus, le_virus)

except FileNotFoundError:
    print(f"❌ ERROR: Model file not found at {model_path}. Please run virus_classifier.py first.")

# ## 4. Anomaly Detection: Autoencoders
# 
# Here, we test our unsupervised autoencoder models. The goal is to see how well they distinguish between benign and malicious traffic by measuring reconstruction error. A good model should have a low detection rate for benign traffic (few false positives) and a high detection rate for malicious traffic (high true positives).

In [ ]:
# ===================================================================
# CELL 5: EVALUATE ANOMALY DETECTION MODELS
# ===================================================================
print("\n" + "=" * 70)
print("Evaluating Anomaly Detection (Autoencoder) Models")
print("=" * 70)

# 1. Load the full test set
df_test_full = load_master_test_set()
features_used = joblib.load(Config.AUTOENCODER_FEATURE_LIST_PATH)

# Separate into benign and malicious traffic, using only the features the AE was trained on
X_test_benign_ae = df_test_full[df_test_full[Config.TARGET_COL] == 'Benign'][features_used].fillna(0)
X_test_malicious_ae = df_test_full[df_test_full[Config.TARGET_COL] == 'Malicious'][features_used].fillna(0)

print(f"✓ Loaded {len(X_test_benign_ae):,} benign and {len(X_test_malicious_ae):,} malicious test samples.")

# --- Evaluate Standard Autoencoder ---
print("\n--- Evaluating: Standard Autoencoder ---")
try:
    input_dim_ae = len(features_used)
    autoencoder = AutoencoderModel(input_dim_ae)
    model_path_ae = os.path.join(Config.MODELS_DIR, 'autoencoder_final.pth')
    autoencoder.load_model(model_path_ae)
    print(f"✓ Model loaded from {model_path_ae}")

    # Scale data using the scaler saved with the model
    X_benign_scaled = autoencoder.scaler.transform(X_test_benign_ae.values)
    X_malicious_scaled = autoencoder.scaler.transform(X_test_malicious_ae.values)
    
    # Create DataLoaders
    benign_loader = DataLoader(IoTDataset(X_benign_scaled), batch_size=Config.AUTOENCODER_BATCH_SIZE)
    malicious_loader = DataLoader(IoTDataset(X_malicious_scaled), batch_size=Config.AUTOENCODER_BATCH_SIZE)

    # Evaluate on BENIGN data (we want a LOW detection rate)
    print("\n--- Results on BENIGN Data (Standard AE) ---")
    benign_preds, _ = autoencoder.detect_anomalies(benign_loader)
    false_positive_rate = benign_preds.sum() / len(benign_preds) * 100 if len(benign_preds) > 0 else 0
    print(f"  🚨 False Positive Rate: {false_positive_rate:.2f}% ({benign_preds.sum():,}/{len(benign_preds):,})")

    # Evaluate on MALICIOUS data (we want a HIGH detection rate)
    print("\n--- Results on MALICIOUS Data (Standard AE) ---")
    malicious_preds, _ = autoencoder.detect_anomalies(malicious_loader)
    detection_rate = malicious_preds.sum() / len(malicious_preds) * 100 if len(malicious_preds) > 0 else 0
    print(f"  🎯 Detection Rate: {detection_rate:.2f}% ({malicious_preds.sum():,}/{len(malicious_preds):,})")

except (FileNotFoundError, AttributeError):
     print(f"❌ ERROR: Model file not found at {model_path_ae} or scaler missing. Please run autoencoder.py first.")

# --- Evaluate Denoising Autoencoder ---
print("\n--- Evaluating: Denoising Autoencoder ---")
try:
    input_dim_dae = len(features_used)
    denoising_ae = DenoisingAutoencoderModel(input_dim_dae)
    model_path_dae = os.path.join(Config.MODELS_DIR, 'denoising_autoencoder_final.pth')
    denoising_ae.load_model(model_path_dae)
    print(f"✓ Model loaded from {model_path_dae}")

    # Scale data (re-using scaled data from above as features are identical)
    # Create DataLoaders
    benign_loader_dae = DataLoader(IoTDataset(X_benign_scaled), batch_size=Config.AUTOENCODER_BATCH_SIZE)
    malicious_loader_dae = DataLoader(IoTDataset(X_malicious_scaled), batch_size=Config.AUTOENCODER_BATCH_SIZE)

    # Evaluate on BENIGN data
    print("\n--- Results on BENIGN Data (Denoising AE) ---")
    benign_preds_dae, _ = denoising_ae.detect_anomalies(benign_loader_dae)
    fp_rate_dae = benign_preds_dae.sum() / len(benign_preds_dae) * 100 if len(benign_preds_dae) > 0 else 0
    print(f"  🚨 False Positive Rate: {fp_rate_dae:.2f}% ({benign_preds_dae.sum():,}/{len(benign_preds_dae):,})")

    # Evaluate on MALICIOUS data
    print("\n--- Results on MALICIOUS Data (Denoising AE) ---")
    malicious_preds_dae, _ = denoising_ae.detect_anomalies(malicious_loader_dae)
    detection_rate_dae = malicious_preds_dae.sum() / len(malicious_preds_dae) * 100 if len(malicious_preds_dae) > 0 else 0
    print(f"  🎯 Detection Rate: {detection_rate_dae:.2f}% ({malicious_preds_dae.sum():,}/{len(malicious_preds_dae):,})")

except (FileNotFoundError, AttributeError):
     print(f"❌ ERROR: Model file not found at {model_path_dae} or scaler missing. Please run autoencoder_denoising.py first.")


# ## 5. Clustering Visualization
# 
# Finally, we apply the trained K-Means model to the malicious traffic in the test set. We then visualize the resulting clusters and color the points by their true malware family labels. This helps us see if the unsupervised clustering algorithm was able to find meaningful, underlying groups in the data that correspond to the actual malware families.

In [ ]:
# %%
# ===================================================================
# CELL 6: VISUALIZE CLUSTERING ON TEST DATA
# ===================================================================
print("\n" + "=" * 70)
print("Visualizing K-Means Clustering on Test Data")
print("=" * 70)

try:
    # 1. Load the clustering model
    kmeans_model = KMeansClustering()
    model_path_kmeans = os.path.join(Config.MODELS_DIR, 'kmeans_clustering.pkl')
    kmeans_model.load_model(model_path_kmeans)
    print(f"✓ Clustering model loaded from {model_path_kmeans}")

    # 2. Prepare malicious data from the test set
    df_test_full = load_master_test_set()
    malicious_test_df = df_test_full[df_test_full[Config.TARGET_COL] == 'Malicious'].copy()
    
    # Use the features the model was trained on
    X_malicious_test = malicious_test_df[kmeans_model.feature_names_in_]

    # 3. Get true labels and encode them
    true_labels_str = malicious_test_df[Config.FAMILY_TARGET_COL].fillna('Unknown').astype(str)
    le_virus_viz = LabelEncoder() # Use a new encoder just for this viz
    true_labels_encoded = le_virus_viz.fit_transform(true_labels_str)
    label_names = le_virus_viz.classes_.tolist()

    print(f"\n✓ Prepared {len(X_malicious_test):,} malicious test samples for visualization.")
    
    # 4. Assign test data to clusters
    cluster_labels = kmeans_model.predict(X_malicious_test)
    kmeans_model.labels = cluster_labels # Set the labels on the object for the visualizer to use

    # 5. Generate and save the visualization
    kmeans_model.visualize_clusters_by_label(
        X_malicious_test,
        true_labels_encoded,
        label_names=label_names,
        save_plot=True
    )
    print("\n✓ Clustering visualization complete and saved to the results directory.")
    
except FileNotFoundError:
    print(f"❌ ERROR: Model file not found at {model_path_kmeans}. Please run clustering.py first.")

# %% [markdown]
# ---
# ## Evaluation Complete
# 
# All models have been evaluated on the master test set. All reports have been printed above, and all visualization artifacts have been saved to the `/results` directory.
# ---